ipw estimators

In [ ]:
#import dbm.sqlite3
import openpyxl
import pandas as pd
import numpy as np
from io import StringIO
import datetime as dt
import econtools
import geopy.distance as geo
import pyreadstat
import pyarrow
import scipy
import pyreadr
import scipy.stats as stats
import statsmodels
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
import os, shutil
from pathlib import Path
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

#use a list
packages = ['math','random','xlrd']
modules = map(__import__,packages)

from datetime import date
print("Today's date is:",date.today())

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

# Load the data
df = pd.read_stata("/Causal_Inference/Treatment_effects/data/cattaneo2.dta",
                      convert_categoricals=False)

# Step 2: Estimate the propensity scores using logistic regression
# 'mbsmoke' is the treatment indicator, and we use 'fage', 'mage', 'mmarried', 'fbaby' as covariates
X = df[['fage', 'mage', 'mmarried', 'fbaby']]  # Covariates for the logistic model
X = sm.add_constant(X)  # Add a constant (intercept) to the model
y = df['mbsmoke']  # Treatment variable

# Fit the logistic regression model
logit_model = sm.Logit(y, X)
logit_result = logit_model.fit()

# Get the propensity scores (probabilities of treatment)
propensity_scores = logit_result.predict(X)

# Step 3: Calculate the inverse probability weights (IPW)
# IPW for treated units: 1 / propensity_score
# IPW for control units: 1 / (1 - propensity_score)
df['ipw'] = np.where(df['mbsmoke'] == 1, 1 / propensity_scores, 1 / (1 - propensity_scores))

# Step 4: Fit a weighted regression model
# Using IPW to estimate ATE via weighted OLS regression
# X_reg = df[['fage', 'mage', 'mmarried', 'fbaby']]  # Covariates for the regression model
X_reg = df[['mbsmoke']]  # Covariates for the regression model (just the treatment variable)
X_reg = sm.add_constant(X_reg)  # Add a constant to the model
y_reg = df['bweight']  # Outcome variable (birth weight)

# Fit the weighted regression model using IPW
weights = df['ipw']
weighted_model = sm.WLS(y_reg, X_reg, weights=weights)
weighted_result = weighted_model.fit()
#weighted_result.summary()

# Step 5: Estimate the Average Treatment Effect (ATE)
# The coefficient for the treatment variable 'mbsmoke' represents the ATE
ate = weighted_result.params['mbsmoke']

# Display the ATE and its confidence interval
ate_confint = weighted_result.conf_int().loc['mbsmoke']

print(f"ATE: {ate}")
print(f"95% Confidence Interval for ATE: {ate_confint}")





In [ ]:
### Example 3 (Stata; R)

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

# Load the data
df = pd.read_stata("/Causal_Inference/Treatment_effects/data/cattaneo2.dta",
                      convert_categoricals=False)

# Step 2: Estimate the propensity scores using logistic regression
# 'mbsmoke' is the treatment indicator, and we use 'fage', 'mage', 'mmarried', 'fbaby' as covariates
X = df[['prenatal1', 'mage', 'mmarried', 'fbaby']]  # Covariates for the logistic model
X = sm.add_constant(X)  # Add a constant (intercept) to the model
y = df['mbsmoke']  # Treatment variable

# Fit the logistic regression model
logit_model = sm.Logit(y, X)
logit_result = logit_model.fit()

# Get the propensity scores (probabilities of treatment)
propensity_scores = logit_result.predict(X)

# Step 3: Calculate the inverse probability weights (IPW)
# IPW for treated units: 1 / propensity_score
# IPW for control units: 1 / (1 - propensity_score)
df['ipw'] = np.where(df['mbsmoke'] == 1, 1 / propensity_scores, 1 / (1 - propensity_scores))

# Step 4: Fit a weighted regression model
# Using IPW to estimate ATE via weighted OLS regression
# X_reg = df[['fage', 'mage', 'mmarried', 'fbaby']]  # Covariates for the regression model
X_reg = df[['mbsmoke']]  # Covariates for the regression model (just the treatment variable)
X_reg = sm.add_constant(X_reg)  # Add a constant to the model
y_reg = df['bweight']  # Outcome variable (birth weight)

# Fit the weighted regression model using IPW
weights = df['ipw']
weighted_model = sm.WLS(y_reg, X_reg, weights=weights)
weighted_result = weighted_model.fit()
#weighted_result.summary()

# Step 5: Estimate the Average Treatment Effect (ATE)
# The coefficient for the treatment variable 'mbsmoke' represents the ATE
ate = weighted_result.params['mbsmoke']

# Display the ATE and its confidence interval
ate_confint = weighted_result.conf_int().loc['mbsmoke']

print(f"ATE: {ate}")
print(f"95% Confidence Interval for ATE: {ate_confint}")

In [ ]:
summary = weighted_result.summary()
print(summary)

In [ ]:
import datetime
print(f"ipw estimators finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")